# Workshop Guide: Declarative Automation Bundles — Deploy RetailHub

Hints for **`lab_09_dabs.ipynb`** (Day 3). Hints only — full answers live in
`notebooks/solution/lab_09_dabs_solution.ipynb`.

## Scenario

> *"Package the RetailHub pipeline + job as a Declarative Automation Bundle and deploy it to a `dev` target — the same way you would promote it to prod from CI."*

The bundle is already written for you in `materials/cicd/` — this lab is about **reading, deploying
and verifying** it, not authoring YAML from scratch.

## Objectives

- Read a `databricks.yml`: bundle name, includes, variables, targets & modes
- (Optional) Create a branch, commit & push and open a pull request from a Git folder
- Override a bundle variable at deploy time
- Run `databricks bundle validate / deploy / run` against a `dev` target
- Verify deployments and run states with the Databricks SDK
- Explain what changes (and what must not) between dev and prod

## Hints per task

### Task 1 — Bundle anatomy
- Everything you need is in the top ~40 lines of `databricks.yml`: `bundle:`, `variables:`, `targets:`.
- "Default target" = the target with `default: true`.
- Count the keys under `variables:` — the names go into the list exactly as written.

### Task 2 — Deploy command
- Skeleton: `databricks bundle deploy -t <target> --var="<name>=<value>"`.
- Your catalog is already in the `CATALOG` variable — build the string with an f-string.
- Keep the quotes around `catalog=...` exactly as in the README.

### Optional — Git folder (before Task 3)
- Branch button next to the Git folder name → **Create Branch** `feature/<your_name>`.
- Small YAML edit (description or `tags:`) → **Commit & Push** → open the PR in the Git provider, not in Databricks.
- Provider not connected? Skip it — no assert depends on it.

### Task 3 — Validate & deploy
- Web terminal: `cd` to `materials/cicd` **inside your Git folder** (`/Workspace/Users/<you>/...`).
- `validate` only parses and resolves — it changes nothing in the workspace; run it freely.
- No CLI? That is what Option C (trainer-driven + observation checklist) is for — tick the boxes as you watch.

### Task 4 — SDK verification
- `WorkspaceClient()` needs no arguments inside a notebook.
- One list comprehension over `w.jobs.list()`; the name lives at `j.settings.name`.
- Dev mode renamed your job — search for the *substring* `retailhub_job`, not an exact match.

### Task 5 — Run state
- `w.jobs.list_runs(job_id=..., limit=5)` returns newest first.
- Two state fields: `life_cycle_state` (is it done?) vs `result_state` (how did it end?).
- `result_state` is `None` until the run terminates — wait and re-run the cell.

### Task 6 — dev vs prod
- Look at the name of your deployed job and at the commented `prod` stub in `databricks.yml`.
- Ask yourself: when you promote to prod, does any *code* change?

## Common pitfalls

| Symptom | Cause / fix |
|---|---|
| `validate` fails with variable errors | Typo in `--var="catalog=..."` — the flag format is exact |
| Deploy targets the trainer catalog | You forgot the `--var` override — the default is `retailhub_trainer` |
| `retailhub_jobs` is empty | Deploy ran in another workspace/profile, or filter string too strict (`[dev …] ` prefix!) |
| `result_state` is `None` | The run is still `RUNNING` — that is a lifecycle state, not a result |
| Permission error on pipeline refresh | Pipeline catalog ≠ your catalog — check the variable override used at deploy time |

← [Lab 09 — DABs](../day3/lab/lab_09_dabs.ipynb) | **[README](../../README.md)**